# Statistical Analysis

In [20]:
import pandas as pd
import duckdb
import os
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import matplotlib.ticker as mticker
import numpy as np

from scipy.stats import mannwhitneyu

conn = duckdb.connect('../database/olist.db')

### Hypothesis Testing

Does late delivery significantly impact review scores?

H0: Review scores for on-time deliveries are NOT significantly higher than late deliveries. 

H1: Review scores for on-time deliveries ARE significantly higher than late deliveries.

Significance level (alpha): 0.05
Test chosen: Mann-Whitney U (one-tailed)
Reason: Review scores are ordinal (1-5), not normally distributed. 

Since our data is not normally distributed we are going to use Mann-Whitney U Test (also known as the Wilcoxon Rank-Sum Test).

In [43]:
query = """ 
    SELECT
        ors.order_id, total_reviews_count, avg_score,
        CASE
            when delivery_days_accuracy <=0 then 1
            else 0
        END as on_time
    FROM order_review_summary ors
    RIGHT JOIN orders_delivery_details odd
    on ors.order_id = odd.order_id
    where order_status = 'delivered'
"""
df = conn.execute(query).df()
# Group 1 - Scores where delivery was on time
group_a = df[df['on_time'] == 1]['avg_score'].dropna()

# Group 2 - Scores where delivery was late
group_b = df[df['on_time'] == 0]['avg_score'].dropna()


print("=" * 50)
print("SANITY CHECK")
print("=" * 50)
print(f"On-time group size     : {len(group_a):,}")
print(f"Late group size        : {len(group_b):,}")
print(f"\nOn-time  — Mean: {group_a.mean():.3f} | Median: {group_a.median():.1f} | Std: {group_a.std():.3f}")
print(f"Late     — Mean: {group_b.mean():.3f} | Median: {group_b.median():.1f} | Std: {group_b.std():.3f}")
print(f"\nSample A (on-time) size : {len(sample_a)}")
print(f"Sample B (late)    size : {len(sample_b)}")


# Taking random sample from each group
sample_a = group_a.sample(n=1000, random_state=48)
sample_b = group_b.sample(n=1000, random_state=48)

# Performing Mann-Whitney U test
stat, p_value = mannwhitneyu(sample_a, sample_b, alternative = 'greater')

alpha = 0.05

print("\n" + "=" * 50)
print("MANN-WHITNEY U TEST RESULTS")
print("=" * 50)
print(f"U Statistic : {stat:,.2f}")
print(f"P-Value     : {p_value:.2e}") 

print("\n" + "=" * 50)
print("INTERPRETATION")
print("=" * 50)

if p_value < alpha:
    print(f"✅ Result    : SIGNIFICANT (p = {p_value:.2e} < alpha = {alpha})")
    print(f"\n📌 Plain English:")
    print(f"   We REJECT the null hypothesis.")
    print(f"   On-time deliveries produce statistically significantly")
    print(f"   higher review scores than late deliveries.")
    print(f"   This result is extremely unlikely to be due to chance.")
else:
    print(f"❌ Result    : NOT SIGNIFICANT (p = {p_value:.2e} > alpha = {alpha})")
    print(f"\n📌 Plain English:")
    print(f"   We FAIL TO REJECT the null hypothesis.")
    print(f"   No statistically significant difference found.")


SANITY CHECK
On-time group size     : 89,443
Late group size        : 6,389

On-time  — Mean: 4.291 | Median: 5.0 | Std: 1.148
Late     — Mean: 2.275 | Median: 1.0 | Std: 1.573

Sample A (on-time) size : 1000
Sample B (late)    size : 1000

MANN-WHITNEY U TEST RESULTS
U Statistic : 808,932.00
P-Value     : 1.15e-139

INTERPRETATION
✅ Result    : SIGNIFICANT (p = 1.15e-139 < alpha = 0.05)

📌 Plain English:
   We REJECT the null hypothesis.
   On-time deliveries produce statistically significantly
   higher review scores than late deliveries.
   This result is extremely unlikely to be due to chance.


A significant p-value only tells you the effect is REAL.
It does NOT tell you how LARGE or MEANINGFUL it is.
With n=1000, almost anything will be significant.
You MUST report effect size.

In [44]:
n1, n2 = len(sample_a), len(sample_b)
effect_size_r = 1 - (2 * stat) / (n1 * n2)  # rank-biserial correlation

print(f"\n" + "=" * 50)
print("EFFECT SIZE — Rank Biserial Correlation (r)")
print("=" * 50)
print(f"r = {effect_size_r:.4f}")

if abs(effect_size_r) < 0.1:
    magnitude = "NEGLIGIBLE"
elif abs(effect_size_r) < 0.3:
    magnitude = "SMALL"
elif abs(effect_size_r) < 0.5:
    magnitude = "MEDIUM"
else:
    magnitude = "LARGE"

print(f"Magnitude   : {magnitude}")
print(f"\n📌 Plain English:")
print(f"   The effect size is {magnitude.lower()}, meaning late delivery")
print(f"   has a {magnitude.lower()} practical impact on review scores.")
print(f"   Statistical significance ≠ Business significance.")


EFFECT SIZE — Rank Biserial Correlation (r)
r = -0.6179
Magnitude   : LARGE

📌 Plain English:
   The effect size is large, meaning late delivery
   has a large practical impact on review scores.
   Statistical significance ≠ Business significance.


TypeError: ufunc 'isnan' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''